In [1]:
BATCH_SIZE = 32
MODEL_SAVE_PATH = "reward_model_6x6_2a"

WIDTH = 9
HEIGHT = 9
NUM_AGENTS = 4
LEARNING_RATE = 0.001
CONV_CHANNELS = [16, 32]
KERNEL_SIZE = 3


In [2]:
import sys
sys.path.append('../../../')
from tadd_helpers.env_functions import State
import pickle
import numpy as np
from models.cnn import CNNDecentralized
from matplotlib import pyplot as plt
from tqdm import tqdm
from pathlib import Path
import torch


--- PyTorch is configured to use: cuda ---


In [3]:
MODEL_SAVE_PATH = Path(MODEL_SAVE_PATH)
if not MODEL_SAVE_PATH.exists():
    MODEL_SAVE_PATH.mkdir(parents=True)


In [4]:
def get_decentralized_reward_when_picked(picked: bool, picker_index, picker_pos, all_agents) -> dict[int, float]:
    """Return decentralized reward of all agents

    Args:
        picked (bool): Whether an agent has picked an apple
        picker_index: _description_
        picker_pos: _description_
        all_agents: dict[int, tuple]: Mapping from agent index to (x, y) position
    """
    res: dict[int, float] = {}
    
    if not picked:
        for agent_idx in all_agents.keys():
            res[agent_idx] = 0.0
        return res

    res[picker_index] = -1

    # Calculate distances from ALL agents to the picker
    all_agent_positions = np.array(list(all_agents.values()))
    distances = np.linalg.norm(all_agent_positions - np.array(picker_pos), axis=1)
    sum_of_distances = np.sum(distances)
    
    for agent_idx in all_agents.keys():
        if agent_idx == picker_index:
            continue
        agent_pos = all_agents[agent_idx]
        agent_distance = np.linalg.norm(np.array(agent_pos) - np.array(picker_pos))
        if sum_of_distances == 0:
            res[agent_idx] = 0.0
        else:
            res[agent_idx] = 2 * agent_distance / sum_of_distances
    return res

def get_picker_index_and_pos_from_state(state: State):
    """Extract the picker index and position from the state. 

    Args:
        state (State): The current environment state
    Returns:
        picked: bool: Whether an agent has picked an apple
        picker_index (int): The index of the picker agent
        picker_pos (tuple): The (x, y) position of the picker agent 
    """
    picked = False
    picker_index = -1
    picker_pos = (-1, -1)
    for agent_idx, agent_pos in state._agents.items():
        # check if the agent_pos is in the apples nd array
        if state._apples[agent_pos] >= 1:
            picked = True
            picker_index = agent_idx
            picker_pos = agent_pos
            break
    return picked, picker_index, picker_pos

def get_decentralized_reward_from_state(state: State) -> dict[int, float]:
    """Calculate the decentralized reward for the self-agent based on the state.

    Args:
        state (State): The current environment state
    Returns:
        dict[int, float]: The decentralized rewards for all agents
    """
    picked, picker_index, picker_pos = get_picker_index_and_pos_from_state(state)
    all_agents = state._agents
    rewards = get_decentralized_reward_when_picked(picked, picker_index, picker_pos, all_agents)
    return rewards

In [7]:
states: list[State] = []
states_file_path = f"centralized_model{WIDTH}x{HEIGHT}_agents{NUM_AGENTS}/trained_states_centralized.pkl"
with open(states_file_path, "rb") as f:
    states = pickle.load(f)

In [8]:
cnns: list[CNNDecentralized] = []
for i in range(NUM_AGENTS):
    model = CNNDecentralized(WIDTH, HEIGHT, LEARNING_RATE, 128, 2, CONV_CHANNELS, KERNEL_SIZE)
    cnns.append(model)

states_to_eval = states
losses = []
for i, state in tqdm(enumerate(states_to_eval), total=len(states_to_eval)):
    rewards = get_decentralized_reward_from_state(state)
    assert len(rewards.values()) == len(cnns), f"Mismatch between rewards and CNN models on state {i}"
    for idx in range(len(rewards)):
        reward = rewards[idx]
        cnns[idx].add_experience_from_raw(cnns[idx].state_to_raw_dict(state), reward, agent_pos=state.agent_position(idx))
        if len(cnns[idx].batch_states) >= BATCH_SIZE:
            loss = cnns[idx].train_batch(BATCH_SIZE)
            losses.append(loss)
        
    

  0%|          | 95/800000 [00:13<32:32:21,  6.83it/s]


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('Training Loss over Time')

In [ ]:
for i in range(2):
    model_path = Path(MODEL_SAVE_PATH / f"model_{i}.pt")
    torch.save(cnns[i].state_dict(), model_path)